# Gemini Batch Jobs — Cost, Time & Token Analysis

Fetches all batch jobs from the Gemini API and breaks down:
- **Job status & timing** (duration, state)
- **Token usage** (prompt / output / thoughts per job and per request)
- **Estimated cost** using Gemini pricing (batch = 50% discount vs. standard)
- **Per-request breakdowns** (error rate, token distribution)

In [ ]:
import os
import sys
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone
from IPython.display import display

from google import genai
from google.genai import types

API_KEY = os.getenv("GOOGLE_AI_WIKI_KEY")
assert API_KEY, "Set the GOOGLE_AI_WIKI_KEY environment variable before running."

client = genai.Client(api_key=API_KEY)
print("Client ready.")

## 1 · Pricing table

Edit the prices (USD per **1 M tokens**) to match the current Gemini pricing page.  
Batch API requests are billed at **50 % of the standard price**.

In [ ]:
# Prices in USD per 1 M tokens (standard / non-batch)
# Source: https://ai.google.dev/gemini-api/docs/pricing  (update as needed)
PRICING = {
    # model-name-fragment -> {input, output}  (all in $/1M tokens)
    "gemini-4-hydrogen": {"input": 0.70, "output": 2.00},
    "gemini-3.8-flash":  {"input": 0.10, "output": 0.40},
    "gemini-2.5-flash":  {"input": 0.10, "output": 0.40},
    "gemini-2.5-pro":    {"input": 1.25, "output": 10.00},
    "gemini-2.0-flash":  {"input": 0.10, "output": 0.40},
    "default":           {"input": 0.10, "output": 0.40},
}

BATCH_DISCOUNT = 0.50   # batch API = 50 % of standard price

def get_price(model_name: str) -> dict:
    model_lower = (model_name or "").lower()
    for key, prices in PRICING.items():
        if key in model_lower:
            return prices
    return PRICING["default"]

def estimate_cost(input_tokens: int, output_tokens: int, model_name: str, batch: bool = True) -> float:
    prices = get_price(model_name)
    cost = (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000
    if batch:
        cost *= BATCH_DISCOUNT
    return cost

print("Pricing table loaded.")

## 2 · Fetch all batch jobs

In [ ]:
all_batches = list(client.batches.list())
print(f"Found {len(all_batches)} batch job(s).")

for b in all_batches:
    print(f"  {b.name}  |  {b.display_name}  |  state={b.state}  |  model={b.model}")

## 3 · Build the jobs summary DataFrame

In [ ]:
def duration_seconds(start, end) -> float | None:
    """Return (end - start) in seconds, handling timezone-naive vs aware."""
    if start is None or end is None:
        return None
    # make both tz-aware (UTC) if they aren't already
    if start.tzinfo is None:
        start = start.replace(tzinfo=timezone.utc)
    if end.tzinfo is None:
        end = end.replace(tzinfo=timezone.utc)
    return (end - start).total_seconds()


rows = []
for b in all_batches:
    # ------- timing -------
    dur = duration_seconds(b.start_time or b.create_time, b.end_time or b.update_time)

    # ------- token totals (summed across all inlined responses) -------
    prompt_tokens = 0
    output_tokens = 0
    thoughts_tokens = 0
    total_tokens = 0
    n_requests = 0
    n_errors = 0

    if b.dest and b.dest.inlined_responses:
        for resp in b.dest.inlined_responses:
            n_requests += 1
            if resp.error is not None:
                n_errors += 1
                continue
            if resp.response and resp.response.usage_metadata:
                u = resp.response.usage_metadata
                prompt_tokens   += u.prompt_token_count    or 0
                output_tokens   += u.candidates_token_count or 0
                thoughts_tokens += u.thoughts_token_count  or 0
                total_tokens    += u.total_token_count     or 0

    cost_usd = estimate_cost(prompt_tokens, output_tokens, b.model or "", batch=True)

    rows.append({
        "name":            b.name,
        "display_name":    b.display_name,
        "model":           b.model,
        "state":           str(b.state),
        "create_time":     b.create_time,
        "start_time":      b.start_time,
        "end_time":        b.end_time,
        "duration_min":    round(dur / 60, 2) if dur is not None else None,
        "n_requests":      n_requests,
        "n_errors":        n_errors,
        "prompt_tokens":   prompt_tokens,
        "output_tokens":   output_tokens,
        "thoughts_tokens": thoughts_tokens,
        "total_tokens":    total_tokens,
        "cost_usd":        round(cost_usd, 6),
    })

jobs_df = pd.DataFrame(rows)
display(jobs_df)

## 4 · Summary statistics

In [ ]:
numeric_cols = ["duration_min", "n_requests", "n_errors",
                "prompt_tokens", "output_tokens", "thoughts_tokens",
                "total_tokens", "cost_usd"]

print("=== Totals across all jobs ===")
totals = jobs_df[numeric_cols].sum(numeric_only=True)
print(totals.to_string())

print("\n=== Per-job summary ===")
display(jobs_df[numeric_cols].describe().round(3))

## 5 · Per-request breakdown (token & error detail)

In [ ]:
req_rows = []

for b in all_batches:
    if not (b.dest and b.dest.inlined_responses):
        continue
    for i, resp in enumerate(b.dest.inlined_responses):
        is_error = resp.error is not None
        pt = ot = tt = tot = 0
        finish_reason = None

        if not is_error and resp.response:
            if resp.response.usage_metadata:
                u = resp.response.usage_metadata
                pt  = u.prompt_token_count     or 0
                ot  = u.candidates_token_count  or 0
                tt  = u.thoughts_token_count    or 0
                tot = u.total_token_count       or 0
            # grab finish reason from the first candidate
            if resp.response.candidates:
                finish_reason = str(resp.response.candidates[0].finish_reason)

        req_rows.append({
            "batch_name":      b.name,
            "display_name":    b.display_name,
            "model":           b.model,
            "request_index":   i,
            "metadata":        str(resp.metadata),
            "is_error":        is_error,
            "error_msg":       str(resp.error) if is_error else None,
            "finish_reason":   finish_reason,
            "prompt_tokens":   pt,
            "output_tokens":   ot,
            "thoughts_tokens": tt,
            "total_tokens":    tot,
            "cost_usd":        round(estimate_cost(pt, ot, b.model or "", batch=True), 8),
        })

req_df = pd.DataFrame(req_rows)

if req_df.empty:
    print("No inlined responses found (jobs may still be running or used GCS/BigQuery output).")
else:
    print(f"Total requests across all jobs: {len(req_df)}")
    print(f"Error rate: {req_df['is_error'].mean()*100:.1f}%")
    display(req_df.head(20))

## 6 · Visualizations

In [ ]:
if jobs_df.empty:
    print("No jobs to plot.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Gemini Batch Jobs — Overview", fontsize=14, fontweight="bold")

    labels = [
        (d[:20] + "…") if d and len(d) > 20 else (d or n.split("/")[-1][-20:])
        for d, n in zip(jobs_df["display_name"], jobs_df["name"])
    ]

    # ---- 1. Duration ----
    ax = axes[0, 0]
    vals = jobs_df["duration_min"].fillna(0)
    ax.barh(labels, vals, color="steelblue")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Job Duration")
    ax.invert_yaxis()

    # ---- 2. Token breakdown per job (stacked bar) ----
    ax = axes[0, 1]
    x = range(len(jobs_df))
    ax.bar(x, jobs_df["prompt_tokens"],   label="Prompt",   color="#4878CF")
    ax.bar(x, jobs_df["output_tokens"],   label="Output",   color="#6ACC65",
           bottom=jobs_df["prompt_tokens"])
    ax.bar(x, jobs_df["thoughts_tokens"], label="Thoughts",  color="#D65F5F",
           bottom=jobs_df["prompt_tokens"] + jobs_df["output_tokens"])
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=7)
    ax.set_ylabel("Tokens")
    ax.set_title("Token Usage per Job")
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.2f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))

    # ---- 3. Estimated cost per job ----
    ax = axes[0, 2]
    ax.barh(labels, jobs_df["cost_usd"], color="#D65F5F")
    ax.set_xlabel("Estimated cost (USD)")
    ax.set_title("Estimated Cost per Job (batch pricing)")
    ax.invert_yaxis()

    # ---- 4. Requests per job + error count ----
    ax = axes[1, 0]
    ax.bar(x, jobs_df["n_requests"], label="Total requests", color="#4878CF", alpha=0.8)
    ax.bar(x, jobs_df["n_errors"],   label="Errors",          color="#D65F5F", alpha=0.9)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=7)
    ax.set_title("Requests per Job")
    ax.legend(fontsize=8)

    # ---- 5. Per-request output token distribution ----
    ax = axes[1, 1]
    if not req_df.empty:
        ok_reqs = req_df[~req_df["is_error"]]
        ax.hist(ok_reqs["output_tokens"], bins=30, color="#6ACC65", edgecolor="white")
        ax.set_xlabel("Output tokens")
        ax.set_ylabel("# requests")
        ax.set_title("Output Token Distribution (per request)")
    else:
        ax.text(0.5, 0.5, "No per-request data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Output Token Distribution")

    # ---- 6. Cumulative cost across jobs (by create_time) ----
    ax = axes[1, 2]
    sorted_jobs = jobs_df.dropna(subset=["create_time"]).sort_values("create_time")
    if not sorted_jobs.empty:
        cumcost = sorted_jobs["cost_usd"].cumsum()
        ax.step(range(len(sorted_jobs)), cumcost, where="post", color="#D65F5F", linewidth=2)
        ax.set_xticks(range(len(sorted_jobs)))
        ax.set_xticklabels(
            [str(t)[:10] if t else "?" for t in sorted_jobs["create_time"]],
            rotation=30, ha="right", fontsize=7
        )
        ax.set_ylabel("Cumulative cost (USD)")
        ax.set_title("Cumulative Spend over Jobs")
    else:
        ax.text(0.5, 0.5, "No create_time data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Cumulative Spend")

    plt.tight_layout()
    plt.savefig("batch_analysis_overview.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Figure saved to batch_analysis_overview.png")

## 7 · Per-request deep-dive (top-N most expensive)

In [ ]:
if not req_df.empty:
    TOP_N = 20

    print(f"=== Top {TOP_N} most expensive individual requests ===")
    top_expensive = req_df.nlargest(TOP_N, "cost_usd")[
        ["display_name", "request_index", "metadata",
         "prompt_tokens", "output_tokens", "cost_usd", "finish_reason"]
    ]
    display(top_expensive)

    print(f"\n=== Errors ===")
    errors = req_df[req_df["is_error"]][["display_name", "request_index", "metadata", "error_msg"]]
    if errors.empty:
        print("No errors 🎉")
    else:
        display(errors)

## 8 · Group by model

In [ ]:
if not jobs_df.empty:
    by_model = (
        jobs_df
        .groupby("model", dropna=False)[["n_requests", "prompt_tokens",
                                          "output_tokens", "total_tokens", "cost_usd"]]
        .sum()
        .sort_values("cost_usd", ascending=False)
    )
    print("=== Cost & tokens grouped by model ===")
    display(by_model)

## 9 · Export to CSV

In [ ]:
jobs_df.to_csv("batch_jobs_summary.csv", index=False)
print("Jobs summary saved → batch_jobs_summary.csv")

if not req_df.empty:
    req_df.to_csv("batch_requests_detail.csv", index=False)
    print("Per-request detail saved → batch_requests_detail.csv")